## Bibliotecas

In [1]:
%pip install pandas -q
%pip install scikit-learn -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import psycopg
import pandas as pd

## Conexão

In [3]:
conn = psycopg.connect(
    host="localhost",
    port=5432,
    dbname="lh_nautical",
    user="postgres",
    password="Admin123"
)

In [4]:
# Consulta SQL para obter os dados de vendas mensais do produto "Bússola de Bordo 702"
query = """
SELECT
    DATE_TRUNC('month', o.created_at)::date AS mes,
    SUM(oi.quantity) AS unidades_vendidas
FROM orders AS o
JOIN order_items AS oi
    ON oi.order_id = o.id
JOIN product_variants AS pv
    ON pv.id = oi.product_variant_id
JOIN products AS p
    ON p.id = pv.product_id
WHERE p.name = 'Bússola de Bordo 702'
  AND o.created_at < '2026-04-01'
GROUP BY DATE_TRUNC('month', o.created_at)
ORDER BY mes;
"""

In [5]:
df = pd.read_sql(query, conn)
df.head()

C:\Users\crisl\AppData\Local\Temp\ipykernel_17060\1821051113.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,mes,unidades_vendidas
0,2020-01-01,31
1,2020-02-01,16
2,2020-03-01,17
3,2020-04-01,32
4,2020-05-01,5


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 74 entries, 0 to 73
Data columns (total 2 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   mes                74 non-null     object
 1   unidades_vendidas  74 non-null     int64 
dtypes: int64(1), object(1)
memory usage: 1.3+ KB


In [7]:
# Converte a coluna "mes" para o tipo datetime
df["mes"] = pd.to_datetime(df["mes"])

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 74 entries, 0 to 73
Data columns (total 2 columns):
 #   Column             Non-Null Count  Dtype        
---  ------             --------------  -----        
 0   mes                74 non-null     datetime64[s]
 1   unidades_vendidas  74 non-null     int64        
dtypes: datetime64[s](1), int64(1)
memory usage: 1.3 KB


## Dados de treino e teste

In [9]:
treino = df[df["mes"] <= "2025-12-01"].copy()

teste = df[
    (df["mes"] >= "2026-01-01") &
    (df["mes"] <= "2026-03-01")
].copy()

In [10]:
print("Treino:", treino["mes"].min(), "até", treino["mes"].max())
print(" Teste:", teste["mes"].min(), "até", teste["mes"].max())

Treino: 2020-01-01 00:00:00 até 2025-12-01 00:00:00
 Teste: 2026-01-01 00:00:00 até 2026-03-01 00:00:00


In [11]:
treino

,mes,unidades_vendidas
0,2020-01-01,31
1,2020-02-01,16
2,2020-03-01,17
3,2020-04-01,32
4,2020-05-01,5
...,...,...
66,2025-08-01,23
67,2025-09-01,31
68,2025-10-01,34
69,2025-11-01,60


In [12]:
teste

,mes,unidades_vendidas
71,2026-01-01,79
72,2026-02-01,68
73,2026-03-01,60


In [13]:
# Calcula a média móvel de 3 meses para o conjunto de treino
treino["media_movel_3"] = (
    treino["unidades_vendidas"]
    .rolling(window=3) 
    .mean()
)

In [14]:
treino.head(5)

,mes,unidades_vendidas,media_movel_3
0,2020-01-01,31,NaN
1,2020-02-01,16,NaN
2,2020-03-01,17,21.333333
3,2020-04-01,32,21.666667
4,2020-05-01,5,18.000000


## Previsão

In [15]:
# Calcula as previsões para os próximos 3 meses usando a média móvel de 3 meses
historico = treino["unidades_vendidas"].tolist() # Transforma a serie pandas em uma lista de valores

previsoes = []

for _ in range(3):
    ultimos_3 = historico[-3:]
    previsao = sum(ultimos_3) / 3
    previsoes.append(previsao)
    historico.append(previsao)

teste["previsao"] = previsoes

teste

,mes,unidades_vendidas,previsao
71,2026-01-01,79,38.666667
72,2026-02-01,68,40.222222
73,2026-03-01,60,33.629630


## MAE (Média do Valor Absoluto dos Erros)

In [16]:
# Avalia o modelo usando o erro absoluto médio (MAE)
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(
    teste["unidades_vendidas"],
    teste["previsao"]
)

print(f"MAE: {mae:.2f}")

MAE: 31.49


In [17]:
previsao_total = round(teste["previsao"].sum())

print("Previsão total para o 1º trimestre de 2026:", previsao_total)

Previsão total para o 1º trimestre de 2026: 113
